# Gold Layer — Fact Table: Sales
## SalesFlow Data Lakehouse | Phase 5: Analytical Layer

Joins `salesflow_dev.silver.orderdetails` and `salesflow_dev.silver.orders`,
enriches with surrogate keys from all four dimensions,
and writes the final fact table to `salesflow_dev.gold.fact_sales`.

**Source tables:**
| Table | Role |
|---|---|
| `salesflow_dev.silver.orderdetails` | Main grain (one row per order line) |
| `salesflow_dev.silver.orders` | Provides `CustomerID`, `EmployeeID`, `OrderDate` |
| `salesflow_dev.gold.dim_customer` | Lookup for `customer_key` |
| `salesflow_dev.gold.dim_product` | Lookup for `product_key` |
| `salesflow_dev.gold.dim_date` | Lookup for `date_key` |
| `salesflow_dev.gold.dim_employee` | Lookup for `employee_key` |

**Design:**
| Column | Type | Description |
|---|---|---|
| `sales_key` | PK | MD5 surrogate key |
| `date_key` | FK | References `dim_date.date_key` |
| `customer_key` | FK | References `dim_customer.customer_key` |
| `product_key` | FK | References `dim_product.product_key` |
| `employee_key` | FK | References `dim_employee.employee_key` |
| `order_id` | DD | Degenerate dimension — natural key from source |
| `quantity` | int | Units ordered |
| `unit_price` | double | Price per unit at time of order |
| `discount` | double | Discount applied (0–1) |
| `line_total` | double | `Quantity × UnitPrice × (1 − Discount)` |
| `load_timestamp` | timestamp | When this record was loaded into Gold |

**Grain:** One row per order line (`OrderID` + `ProductID`)

In [0]:
%run ../04_Utils/common_functions

## 1. Read Silver Sources (VALID records only)

In [0]:
from pyspark.sql.functions import current_timestamp, col, date_format

# Read only VALID order details from Silver
df_details = spark.table("salesflow_dev.silver.orderdetails") \
                  .filter(col("data_quality_status") == "VALID")

# Read only VALID orders from Silver
df_orders = spark.table("salesflow_dev.silver.orders") \
                 .filter(col("data_quality_status") == "VALID")

print(f"Valid order details: {df_details.count()}")
print(f"Valid orders: {df_orders.count()}")

## 2. Read Dimension Tables
Load only the columns needed for the surrogate key lookups.

In [0]:
# Load only PK + NK from each dimension — minimizes shuffle during joins
dim_customer = spark.table("salesflow_dev.gold.dim_customer") \
                    .select("customer_key", "customer_id")

dim_product = spark.table("salesflow_dev.gold.dim_product") \
                   .select("product_key", "product_id")

dim_date = spark.table("salesflow_dev.gold.dim_date") \
               .select("date_key", "full_date")

dim_employee = spark.table("salesflow_dev.gold.dim_employee") \
                    .select("employee_key", "employee_id")

print("Dimensions loaded:")
print(f"  dim_customer : {dim_customer.count()} rows")
print(f"  dim_product  : {dim_product.count()} rows")
print(f"  dim_date     : {dim_date.count()} rows")
print(f"  dim_employee : {dim_employee.count()} rows")

## 3. Join Order Details with Orders
Bring in `CustomerID`, `EmployeeID`, and `OrderDate` from the orders table.
These are needed to resolve the dimension surrogate keys.

In [0]:
from pyspark.sql.functions import col

# Cast OrderID to string in both DataFrames to ensure type consistency on join
df_details = df_details.withColumn("OrderID", col("OrderID").cast("string"))

df_orders_slim = df_orders.select(
    col("OrderID").cast("string"),
    col("CustomerID"),
    col("EmployeeID"),
    col("OrderDate")
)

# Join order details with orders to get the header-level attributes
df = df_details.join(
    df_orders_slim,
    on="OrderID",
    how="inner"  # inner: only lines with a valid order header
)

print(f"Records after details + orders join: {df.count()}")

In [0]:
# Join order details with orders to get the header-level attributes
df = df_details.join(
    df_orders.select("OrderID", "CustomerID", "EmployeeID", "OrderDate"),
    on="OrderID",
    how="inner"  # inner: only lines with a valid order header
)

print(f"Records after details + orders join: {df.count()}")

## 4. Resolve Surrogate Keys from Dimensions

Joins with each dimension to replace natural keys with surrogate keys.  
All joins are **left** to preserve fact records even if a dimension member
is missing — unresolved keys will be `null`, which is preferable to
silently losing sales records.

In [0]:
# 4.1 Resolve customer_key
df = df.join(
    dim_customer,
    df["CustomerID"] == dim_customer["customer_id"],
    how="left"
).drop("customer_id")

# 4.2 Resolve product_key
df = df.join(
    dim_product,
    df["ProductID"] == dim_product["product_id"],
    how="left"
).drop("product_id")

# 4.3 Resolve date_key — match OrderDate to full_date in dim_date
df = df.join(
    dim_date,
    df["OrderDate"] == dim_date["full_date"],
    how="left"
).drop("full_date")

# 4.4 Resolve employee_key
df = df.join(
    dim_employee,
    df["EmployeeID"] == dim_employee["employee_id"],
    how="left"
).drop("employee_id")

print(f"Records after dimension lookups: {df.count()}")

# Sanity check — how many records have unresolved keys
print("\nUnresolved keys check:")
print(f"  Missing customer_key : {df.filter(col('customer_key').isNull()).count()}")
print(f"  Missing product_key  : {df.filter(col('product_key').isNull()).count()}")
print(f"  Missing date_key     : {df.filter(col('date_key').isNull()).count()}")
print(f"  Missing employee_key : {df.filter(col('employee_key').isNull()).count()}")

## 5. Select and Rename Columns
Keep only fact columns — drop all Silver metadata and natural keys
that have already been replaced by surrogate keys.

In [0]:
# Select final fact columns and rename to snake_case
df = df.select(
    col("OrderID").alias("order_id"),
    col("date_key"),
    col("customer_key"),
    col("product_key"),
    col("employee_key"),
    col("Quantity").alias("quantity"),
    col("UnitPrice").alias("unit_price"),
    col("Discount").alias("discount"),
    col("line_total")
)

## 6. Add Surrogate Key
Generates `sales_key` as an MD5 hash of `order_id` + `product_key` —
the natural composite grain of the fact table.

In [0]:
# Composite key: order_id + product_key uniquely identifies each sales line
df = add_surrogate_key(df, "sales", ["order_id", "product_key"])

## 7. Add Load Timestamp

In [0]:
# load_timestamp captures when this record was written to Gold
df = df.withColumn("load_timestamp", current_timestamp())

## 8. Final Column Order
PK first, FKs second, degenerate dimension, measures, metadata last.

In [0]:
df = df.select(
    "sales_key",
    "date_key",
    "customer_key",
    "product_key",
    "employee_key",
    "order_id",
    "quantity",
    "unit_price",
    "discount",
    "line_total",
    "load_timestamp"
)

print(f"Total records in fact table: {df.count()}")
display(df.limit(5))

## 9. Save as Delta Table

In [0]:
# Write to Gold layer as Delta table — overwrite for first load
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("salesflow_dev.gold.fact_sales")

print("Table saved: salesflow_dev.gold.fact_sales")

## 10. Validation

In [0]:
fact_sales = spark.table("salesflow_dev.gold.fact_sales")

# Record count
print(f"Total records: {fact_sales.count()}")

# Surrogate key uniqueness — must be 0 duplicates
duplicate_keys = fact_sales.groupBy("sales_key").count().filter(col("count") > 1)
print(f"\nDuplicate sales_keys (expected 0): {duplicate_keys.count()}")

# Unresolved foreign keys — should all be 0
print("\nUnresolved foreign keys:")
print(f"  Missing date_key     : {fact_sales.filter(col('date_key').isNull()).count()}")
print(f"  Missing customer_key : {fact_sales.filter(col('customer_key').isNull()).count()}")
print(f"  Missing product_key  : {fact_sales.filter(col('product_key').isNull()).count()}")
print(f"  Missing employee_key : {fact_sales.filter(col('employee_key').isNull()).count()}")

# Revenue sanity check
print("\nLine total statistics:")
display(fact_sales.select("line_total").summary())

# Total revenue
from pyspark.sql.functions import sum as spark_sum, round as spark_round
total_revenue = fact_sales.agg(
    spark_round(spark_sum("line_total"), 2).alias("total_revenue")
)
print("\nTotal revenue:")
display(total_revenue)

# Schema
print("\nSchema:")
fact_sales.printSchema()

# Sample
print("\nFirst 5 rows:")
display(fact_sales.limit(5))